In [1]:
#datakit provide pf inputdata

In [12]:
from gridfm_datakit.interactive import interactive_interface
interactive_interface()

Text(value='user_config.yaml', description='Config Filename:', layout=Layout(width='400px'), placeholder='e.g.…

Button(button_style='info', description='Create YAML Config', style=ButtonStyle(), tooltip='Save the current c…

Button(button_style='success', description='Generate and Run Configuration', style=ButtonStyle(), tooltip='Cli…

In [ ]:
/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24

In [1]:
import pandas as pd, numpy as np

pf_path="/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24/case24_ieee_rts/raw/pf_node.csv"
out_path="/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24/case24_ieee_rts/raw/pf_node_missing.csv"

df=pd.read_csv(pf_path)
rng=np.random.default_rng(42)

prefer=["Vm","Va","Pg","Qg","Pf","Qf","vm","va","pg","qg","pf","qf","p_mw","q_mvar"]
cols=[c for c in prefer if c in df.columns]
if len(cols)<6:
    num=[c for c in df.select_dtypes(include=[np.number]).columns if c not in ["scenario","bus"]]
    cols=(cols+num)[:6]
else:
    cols=cols[:6]

vals=df[cols].to_numpy(copy=True).astype(float)
mask=rng.random(vals.shape)<0.3
vals[mask]= np.nan
df.loc[:,cols]=vals

df.to_csv(out_path,index=False)
print("masked_cols:",cols)
print("saved:",out_path)


masked_cols: ['Vm', 'Va', 'Pg', 'Qg', 'Pd', 'Qd']
saved: /Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24/case24_ieee_rts/raw/pf_node_missing.csv


In [2]:
# 读取datakit create的pf文件

In [1]:
import warnings
import os
import time
import copy
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
from owlready2 import *
from pypower.api import case24_ieee_rts
from pypower.api import case30
from pypower.api import case118

ppc_base = case24_ieee_rts()


warnings.filterwarnings("ignore")

In [2]:
onto = get_ontology("file:///Users/hhujunqi/Desktop/OPF_Ontology.owl").load()
df = pd.read_csv("/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24/case24_ieee_rts/raw/pf_node_missing.csv")
edge_params = pd.read_csv("/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24/case24_ieee_rts/raw/edge_params.csv")
branch_df = pd.read_csv("/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24/case24_ieee_rts/raw/branch_idx_removed.csv")

In [3]:
prefer = "fast"  # accurate / fast / unknown
g = onto.world.as_rdflib_graph()

In [4]:
def get_runtime_state(onto):
    MS = onto.ModelSelectionstate
    inst = list(MS.instances())
    return inst[0] if inst else MS("runtime_state")

def fill_has_flags(onto, df):
    MS = onto.ModelSelectionstate
    s = get_runtime_state(onto)
    cols = set(df.columns)

    for dp in onto.data_properties():
        if dp.domain and MS not in dp.domain:
            continue
        if dp.range and bool not in dp.range:
            continue
        if dp.name.startswith("has"):
            col = dp.name[3:]
            ok = (col in df.columns) and (not df[col].isna().any())
            setattr(s, dp.name, [bool(ok)])

    return s

def select_model_from_owl(onto, df):
    global prefer, g

    s = fill_has_flags(onto, df)

    s.preferAccurate = [prefer == "accurate"]
    s.preferFast     = [prefer == "fast"]
    s.preferUnknown  = [prefer == "unknown"]
    
    q = """
PREFIX : <http://www.semanticweb.org/hhujunqi/OPF_Ontology#>

SELECT ?m ?rank WHERE {
  :CurrentState a :ModelSelectionstate ;
    :hasPd ?hasPd ; :hasQd ?hasQd ; :hasVm ?hasVm ; :hasVa ?hasVa ;
    :preferFast ?pf ; :preferAccurate ?pa ; :preferUnknown ?pu .

  VALUES ?m { :Model_ACOPF :model_GridFM :model_GPTOPF }

  OPTIONAL { ?m :requiresPd ?rPd . }  BIND(COALESCE(?rPd,false) AS ?needPd)
  OPTIONAL { ?m :requiresQd ?rQd . }  BIND(COALESCE(?rQd,false) AS ?needQd)
  OPTIONAL { ?m :requiresVm ?rVm . }  BIND(COALESCE(?rVm,false) AS ?needVm)
  OPTIONAL { ?m :requiresVa ?rVa . }  BIND(COALESCE(?rVa,false) AS ?needVa)

  FILTER((! ?needPd) || ?hasPd)
  FILTER((! ?needQd) || ?hasQd)
  FILTER((! ?needVm) || ?hasVm)
  FILTER((! ?needVa) || ?hasVa)

  OPTIONAL { ?m :rankPreferAccurate ?ra . }
  OPTIONAL { ?m :rankPreferFast ?rf . }
  OPTIONAL { ?m :rankPreferUnknown ?ru . }

  BIND(
    IF(?pa, COALESCE(?ra, 999),
      IF(?pf, COALESCE(?rf, 999),
              COALESCE(?ru, 999)
      )
    ) AS ?rank
  )
}
ORDER BY ?rank
LIMIT 1

"""
    rows = list(g.query(q))
    if not rows:
        return None
    return str(rows[0][0]).split("#")[-1]


In [5]:
model = select_model_from_owl(onto, df)
print("prefer =", prefer, "| model =", model)

prefer = fast | model = model_GridFM


In [6]:
def run_acopf(df):

    SAVE_DIR = "/Users/hhujunqi/Desktop/HHOPF/ACOPF_FINAL/case24"
    os.makedirs(SAVE_DIR, exist_ok=True)

    pf = df.copy()
    pf.columns = [c.lower() for c in pf.columns]
    pf["scenario"] = pf["scenario"].astype(int)
    pf["bus"] = pf["bus"].astype(int)

    col_pd = "pd" if "pd" in pf.columns else ("pd_mw" if "pd_mw" in pf.columns else None)
    col_qd = "qd" if "qd" in pf.columns else ("qd_mvar" if "qd_mvar" in pf.columns else None)
    if col_pd is None or col_qd is None:
        raise ValueError(f"pf 缺少 pd/qd 列: {pf.columns.tolist()}")


    bd = branch_df.copy()
    bd.columns = [c.lower() for c in bd.columns]

    rm_long = (
        bd
        .melt(id_vars=["scenario"], value_name="branch_idx")
        .dropna(subset=["branch_idx"])
    )
    rm_long["scenario"] = rm_long["scenario"].astype(int)
    rm_long["branch_idx"] = rm_long["branch_idx"].astype(int)


    net_base = pn.case24_ieee_rts()
    n_lines = len(net_base.line)
    n_trafos = len(net_base.trafo)
    nbus = len(net_base.bus)


    bmin, bmax = int(pf["bus"].min()), int(pf["bus"].max())
    bus_shift = 1 if (bmin == 1 and bmax == nbus) else 0


    opf_results, node_data, gen_data, edge_data = [], [], [], []
    time_results = []  
    scenarios = sorted(pf["scenario"].unique())

    for sid in scenarios:
        t0 = time.time()   
        net = copy.deepcopy(net_base)
        pf_case = pf[pf["scenario"] == sid]


        net.load.drop(net.load.index, inplace=True)
        for _, row in pf_case.iterrows():
            tb = int(row["bus"]) - bus_shift
            if not (0 <= tb < nbus):
                raise ValueError(
                    f"Bus index out of range: raw={int(row['bus'])}, shift={bus_shift}, mapped={tb}"
                )
            pp.create_load(
                net,
                bus=tb,
                p_mw=float(row[col_pd]),
                q_mvar=float(row[col_qd])
            )


        rm_indices = sorted(set(
            rm_long.loc[rm_long["scenario"] == sid, "branch_idx"].tolist()
        ))
        for b_idx in rm_indices:
            if b_idx < n_lines:
                net.line.at[b_idx, "in_service"] = False
            else:
                trafo_idx = b_idx - n_lines
                if 0 <= trafo_idx < n_trafos:
                    net.trafo.at[trafo_idx, "in_service"] = False

        # ACOPF
        try:
            pp.runopp(
                net,
                calculate_voltage_angles=True,
                enforce_q_lims=True,
                suppress_warnings=True
            )
            success = bool(net.OPF_converged)
        except:
            success = False

        cost_val = np.nan
        if success:
            try:
                cost_val = (
                    float(net.res_cost["cost"].sum())
                    if isinstance(net.res_cost, pd.DataFrame)
                    else float(net.res_cost)
                )
            except:
                cost_val = np.nan
        elapsed = time.time() - t0   
        print(f"Scenario {sid:4d} | Success: {int(success)} | Cost: {cost_val}")

        opf_results.append([sid, cost_val, int(success)])
        time_results.append([sid, elapsed])  

        if success:
            bn = net.res_bus.copy();  bn["bus"] = bn.index;  bn["scenario"] = sid;  node_data.append(bn)
            gn = net.res_gen.copy();  gn["gen"] = gn.index;  gn["scenario"] = sid;  gen_data.append(gn)
            ln = net.res_line.copy(); ln["line"] = ln.index; ln["scenario"] = sid;  edge_data.append(ln)


    pd.DataFrame(
        opf_results,
        columns=["scenario", "total_cost", "success"]
    ).to_csv(os.path.join(SAVE_DIR, "opf_cost.csv"), index=False)

    pd.DataFrame(
        time_results,
        columns=["scenario", "time_sec"]
    ).to_csv(os.path.join(SAVE_DIR, "opf_time.csv"), index=False)


    if node_data:
        pd.concat(node_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_node.csv"), index=False
        )
    if gen_data:
        pd.concat(gen_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_gen.csv"), index=False
        )
    if edge_data:
        pd.concat(edge_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_edge.csv"), index=False
        )

    return {
        "model": "Model_ACOPF",
        "save_dir": SAVE_DIR,
        "n_scenarios": len(scenarios)
    }

In [7]:
def run_model_gridfm_predict_then_opf():

    import sys, os, shutil
    import pandas as pd
    from gridfm_graphkit.__main__ import main

    sys.argv = [
        "gridfm_graphkit",
        "predict",
        "--config", "/Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/case24_ieee_rts_base.yaml",
        "--data_path", "/Users/hhujunqi/Desktop/HHOPF/DATAKITEND/case24",
        "--log_dir", "/Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/case24/mlruns",
        "--model_path", "/Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/GridFM_v0_2_.pth",
        "--output_path", "/Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/case24",
        "--exp_name", "case24_exp",
        "--run_name", "predict_run",
    ]
    main()

    df_pred = pd.read_csv("/Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/case24/predictions.csv")

    print("Prediction loaded:", df_pred.shape)

    df_pred = df_pred.rename(columns={"Pd": "PD", "Qd": "QD", "Vm": "VM", "Va": "VA"})
    return run_gridfm(df_pred, branch_df)

In [10]:
def run_gridfm(df_pred, branch_df):

    SAVE_DIR = "/Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/case24"
    os.makedirs(SAVE_DIR, exist_ok=True)

    pf = df_pred.copy()
    pf.columns = [c.lower() for c in pf.columns]
    pf["scenario"] = pf["scenario"].astype(int)
    pf["bus"] = pf["bus"].astype(int)

    col_pd = "pd" if "pd" in pf.columns else ("pd_mw" if "pd_mw" in pf.columns else None)
    col_qd = "qd" if "qd" in pf.columns else ("qd_mvar" if "qd_mvar" in pf.columns else None)
    if col_pd is None or col_qd is None:
        raise ValueError(f"pf 缺少 pd/qd 列: {pf.columns.tolist()}")


    bd = branch_df.copy()
    bd.columns = [c.lower() for c in bd.columns]

    rm_long = (
        bd
        .melt(id_vars=["scenario"], value_name="branch_idx")
        .dropna(subset=["branch_idx"])
    )
    rm_long["scenario"] = rm_long["scenario"].astype(int)
    rm_long["branch_idx"] = rm_long["branch_idx"].astype(int)


    net_base = pn.case24_ieee_rts()
    n_lines = len(net_base.line)
    n_trafos = len(net_base.trafo)
    nbus = len(net_base.bus)


    bmin, bmax = int(pf["bus"].min()), int(pf["bus"].max())
    bus_shift = 1 if (bmin == 1 and bmax == nbus) else 0


    opf_results, node_data, gen_data, edge_data = [], [], [], []
    time_results = []  
    scenarios = sorted(pf["scenario"].unique())

    for sid in scenarios:
        t0 = time.time()  
        net = copy.deepcopy(net_base)
        pf_case = pf[pf["scenario"] == sid]

        # 负荷
        net.load.drop(net.load.index, inplace=True)
        for _, row in pf_case.iterrows():
            tb = int(row["bus"]) - bus_shift
            if not (0 <= tb < nbus):
                raise ValueError(
                    f"Bus index out of range: raw={int(row['bus'])}, shift={bus_shift}, mapped={tb}"
                )
            pp.create_load(
                net,
                bus=tb,
                p_mw=float(row[col_pd]),
                q_mvar=float(row[col_qd])
            )


        rm_indices = sorted(set(
            rm_long.loc[rm_long["scenario"] == sid, "branch_idx"].tolist()
        ))
        for b_idx in rm_indices:
            if b_idx < n_lines:
                net.line.at[b_idx, "in_service"] = False
            else:
                trafo_idx = b_idx - n_lines
                if 0 <= trafo_idx < n_trafos:
                    net.trafo.at[trafo_idx, "in_service"] = False


        try:
            pp.runopp(
                net,
                calculate_voltage_angles=True,
                enforce_q_lims=True,
                suppress_warnings=True
            )
            success = bool(net.OPF_converged)
        except:
            success = False

        cost_val = np.nan
        if success:
            try:
                cost_val = (
                    float(net.res_cost["cost"].sum())
                    if isinstance(net.res_cost, pd.DataFrame)
                    else float(net.res_cost)
                )
            except:
                cost_val = np.nan
        elapsed = time.time() - t0  
        print(f"Scenario {sid:4d} | Success: {int(success)} | Cost: {cost_val}")

        opf_results.append([sid, cost_val, int(success)])
        time_results.append([sid, elapsed]) 

        if success:
            bn = net.res_bus.copy();  bn["bus"] = bn.index;  bn["scenario"] = sid;  node_data.append(bn)
            gn = net.res_gen.copy();  gn["gen"] = gn.index;  gn["scenario"] = sid;  gen_data.append(gn)
            ln = net.res_line.copy(); ln["line"] = ln.index; ln["scenario"] = sid;  edge_data.append(ln)


    pd.DataFrame(
        opf_results,
        columns=["scenario", "total_cost", "success"]
    ).to_csv(os.path.join(SAVE_DIR, "opf_cost.csv"), index=False)

    pd.DataFrame(
        time_results,
        columns=["scenario", "time_sec"]
    ).to_csv(os.path.join(SAVE_DIR, "opf_time.csv"), index=False)


    if node_data:
        pd.concat(node_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_node.csv"), index=False
        )
    if gen_data:
        pd.concat(gen_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_gen.csv"), index=False
        )
    if edge_data:
        pd.concat(edge_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_edge.csv"), index=False
        )

    return {
        "model": "Model_GRIDFMOPF",
        "save_dir": SAVE_DIR,
        "n_scenarios": len(scenarios)
    }

In [8]:
import os, copy, time
import pandas as pd
import pandapower.networks as pn
import pandapower as pp

def run_gridfm(df_pred, branch_df):

    SAVE_DIR = "/Users/hhujunqi/Desktop/HHOPF/GRIDFMOPF_FINAL_missing/case24"
    os.makedirs(SAVE_DIR, exist_ok=True)

    net_base = pn.case24_ieee_rts()


    bd = branch_df.copy()
    bd.columns = [c.lower() for c in bd.columns]
    rm_long = (
        bd.melt(id_vars=["scenario"], value_name="branch_idx")
          .dropna(subset=["branch_idx"])
    )
    rm_long["scenario"] = rm_long["scenario"].astype(int)
    rm_long["branch_idx"] = rm_long["branch_idx"].astype(int)

    opf_results, node_data, gen_data, edge_data = [], [], [], []
    time_results = []

    scens = sorted(df_pred["scenario"].unique())

    for sid in scens:
        t0 = time.time()
        df_s = df_pred[df_pred["scenario"] == sid]
        net = copy.deepcopy(net_base)


        net.load.drop(net.load.index, inplace=True)

        df_load = (
            df_s.groupby("bus", as_index=False)
                .agg({"PD": "sum", "QD": "sum"})
        )

        for r in df_load.itertuples(index=False):
            pp.create_load(
                net,
                bus=int(r.bus),
                p_mw=float(r.PD),
                q_mvar=float(r.QD)
            )

        if {"VM", "VA"}.issubset(df_s.columns):
            for r in df_s.itertuples(index=False):
                b_idx = int(r.bus)
                if b_idx in net.bus.index:
                    net.bus.at[b_idx, "vm_pu"] = float(r.VM)
                    net.bus.at[b_idx, "va_degree"] = float(r.VA)


        rm_indices = rm_long.loc[
            rm_long["scenario"] == int(sid),
            "branch_idx"
        ].tolist()

        for b_idx in rm_indices:
            if b_idx < len(net.line):
                net.line.at[b_idx, "in_service"] = False
            else:
                t_idx = b_idx - len(net.line)
                if 0 <= t_idx < len(net.trafo):
                    net.trafo.at[t_idx, "in_service"] = False

        success, cost_val = 0, float("nan")

        try:
            # PF
            pp.runpp(
                net,
                calculate_voltage_angles=True,
                init="auto"
            )

            # OPF 第一次尝试
            try:
                pp.runopp(
                    net,
                    calculate_voltage_angles=True,
                    enforce_q_lims=True,
                    init="pf"
                )
            except Exception:

                pp.runopp(
                    net,
                    calculate_voltage_angles=True,
                    enforce_q_lims=False,
                    init="pf"
                )

            if getattr(net, "OPF_converged", False):
                success = 1
                cost_val = float(net.res_cost)
                print("sid:", sid,
                   "load_sum:", net.load.p_mw.sum(),
                   "gen_sum:", net.res_gen.p_mw.sum() if len(net.res_gen) else None,
                   "ext_grid_sum:", net.res_ext_grid.p_mw.sum() if "res_ext_grid" in net else None,
                   "poly_cost:", len(net.poly_cost),
                   "pwl_cost:", len(net.pwl_cost))
                print("poly_cost head:")
                print(net.poly_cost.head())

                print("gen head:")
                print(net.gen.head())

                print("ext_grid head:")
                print(net.ext_grid.head())
        except Exception as e:
            # 不 raise，直接记失败继续
            print(f"Scenario {sid} failed:", e)

        elapsed = time.time() - t0
        opf_results.append([int(sid), cost_val, int(success)])
        time_results.append([int(sid), elapsed])

        if success:
            for res_table, storage, name in [
                (net.res_bus, node_data, "bus"),
                (net.res_gen, gen_data, "gen"),
                (net.res_line, edge_data, "line"),
            ]:
                temp_df = res_table.copy()
                temp_df["scenario"] = int(sid)
                temp_df[name] = temp_df.index
                storage.append(temp_df)


    pd.DataFrame(
        opf_results,
        columns=["scenario", "total_cost", "success"]
    ).to_csv(os.path.join(SAVE_DIR, "opf_cost.csv"), index=False)

    pd.DataFrame(
        time_results,
        columns=["scenario", "time_sec"]
    ).to_csv(os.path.join(SAVE_DIR, "opf_time.csv"), index=False)

    if node_data:
        pd.concat(node_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_node.csv"),
            index=False
        )

    if gen_data:
        pd.concat(gen_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_gen.csv"),
            index=False
        )

    if edge_data:
        pd.concat(edge_data, ignore_index=True).to_csv(
            os.path.join(SAVE_DIR, "opf_edge.csv"),
            index=False
        )

    return {"status": "success", "scenarios_processed": len(scens)}


In [9]:
MODEL_RUNNER = {
    "Model_ACOPF": run_acopf,
    "model_GridFM": lambda _: run_model_gridfm_predict_then_opf(),
}

result = MODEL_RUNNER[model](df)
print("result:", result)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Experiment with name case24_exp not found. Creating it.


Setting seed to 42
Loading model weights from /Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/GridFM_v0_2_.pth


Processing...
100%|███████████████████████████████████████| 889/889 [00:00<00:00, 1078.96it/s]
Done!


Predicting: |                                             | 0/? [00:00<?, ?it/s]

Saved predictions to /Users/hhujunqi/Desktop/HHOPF/GRIDFM_FINAL_missing/case24/predictions.csv
Prediction loaded: (21312, 11)
sid: 0 load_sum: 1243.61670666 gen_sum: 413.9470088233637 ext_grid_sum: 69.00000000002474 poly_cost: 33 pwl_cost: 0
poly_cost head:
   element        et   cp0_eur  cp1_eur_per_mw  cp2_eur_per_mw2  cq0_eur  \
0        0       gen  400.6849        130.0000         0.000000      0.0   
1        0      sgen  400.6849        130.0000         0.000000      0.0   
2        7      sgen  781.5210         43.6615         0.052672      0.0   
3        0  ext_grid  832.7575         48.5804         0.007170      0.0   
4        8      sgen  832.7575         48.5804         0.007170      0.0   

   cq1_eur_per_mvar  cq2_eur_per_mvar2  
0               0.0                0.0  
1               0.0                0.0  
2               0.0                0.0  
3               0.0                0.0  
4               0.0                0.0  
gen head:
   name  bus  p_mw  vm_pu  sn

In [9]:
from openai import OpenAI
import os
client = OpenAI(
    api_key="A",  
)

In [10]:
def run_gptopf(df):
    import re
    import numpy as np
    import pandas as pd
    import pandapower.networks as nw
    import os
    SAVE_DIR = "/Users/hhujunqi/Desktop/HHOPF/GPTOPF_FINAL/case24"
    os.makedirs(SAVE_DIR, exist_ok=True)

    if "client" not in globals():
        raise NameError("缺少全局变量 client")

    client_ = globals()["client"]
    edge_params_ = globals().get("edge_params", None)
    branch_removed_ = globals().get("branch_df", None)

    # ===== PF =====
    pf = df.copy()
    if "scenario" not in pf.columns:
        raise ValueError(f"df 缺少 scenario 列，当前列={pf.columns.tolist()}")
    if "Pd" not in pf.columns:
        raise ValueError(f"df 缺少 Pd 列，当前列={pf.columns.tolist()}")
    pf["scenario"] = pf["scenario"].astype(int)

    # ===== network + bounds + cost coef =====
    net = pn.case24_ieee_rts()

    gen_min = net.gen["min_p_mw"].to_numpy(dtype=float)
    gen_max = net.gen["max_p_mw"].to_numpy(dtype=float)

    sgen_min = net.sgen["min_p_mw"].to_numpy(dtype=float) if len(net.sgen) else np.array([], dtype=float)
    sgen_max = net.sgen["max_p_mw"].to_numpy(dtype=float) if len(net.sgen) else np.array([], dtype=float)

    ext_min = net.ext_grid["min_p_mw"].to_numpy(dtype=float) if ("min_p_mw" in net.ext_grid.columns) else np.array([], dtype=float)
    ext_max = net.ext_grid["max_p_mw"].to_numpy(dtype=float) if ("max_p_mw" in net.ext_grid.columns) else np.array([], dtype=float)

    output_min = np.concatenate([gen_min, sgen_min, ext_min])
    output_max = np.concatenate([gen_max, sgen_max, ext_max])
    units = int(len(output_min))
    if units == 0:
        raise ValueError("units=0：net 里没有可控出力(gen/sgen/ext_grid)。")

    if not hasattr(net, "poly_cost") or net.poly_cost is None or len(net.poly_cost) == 0:
        raise ValueError("net.poly_cost 为空：case24_ieee_rts() 没有成本表。")

    coef_A = net.poly_cost["cp2_eur_per_mw2"].to_numpy(dtype=float)
    coef_B = net.poly_cost["cp1_eur_per_mw"].to_numpy(dtype=float)
    coef_C = net.poly_cost["cp0_eur"].to_numpy(dtype=float)

    if len(coef_A) != units or len(coef_B) != units or len(coef_C) != units:
        raise ValueError(f"poly_cost 行数({len(net.poly_cost)})与 units({units})不一致，先对齐成本表。")

    removed_map = {}
    n_branch = 0
    if edge_params_ is not None:
        ep = edge_params_.copy()
        for c in ["from_bus", "to_bus", "rate_a"]:
            if c not in ep.columns:
                raise ValueError(f"edge_params 缺列 {c}，当前列={ep.columns.tolist()}")
        from_bus = ep["from_bus"].astype(int).to_numpy()
        to_bus   = ep["to_bus"].astype(int).to_numpy()
        rate_a   = ep["rate_a"].astype(float).to_numpy()
        n_branch = int(len(ep))
    else:
        from_bus = to_bus = rate_a = np.array([])
        n_branch = 0

    if branch_removed_ is not None:
        br = branch_removed_.copy()
        if "scenario" not in br.columns:
            raise ValueError(f"branch_df 缺列 scenario，当前列={br.columns.tolist()}")
        br["scenario"] = br["scenario"].astype(int)
        idx_cols = [c for c in br.columns if c != "scenario"]

        for _, row in br.iterrows():
            sid = int(row["scenario"])
            removed_idxs = [int(v) for v in row[idx_cols].values if not pd.isna(v)]
            if removed_idxs:
                removed_map[sid] = removed_idxs

    def get_active_branch_mask(sid: int) -> np.ndarray:
        if n_branch == 0:
            return np.array([], dtype=bool)
        mask = np.ones(n_branch, dtype=bool)
        for idx in removed_map.get(int(sid), []):
            if 0 <= idx < n_branch:
                mask[idx] = False
        return mask

    load_demand_map = pf.groupby("scenario")["Pd"].sum().astype(float).to_dict()

    def calculate(thermal_output: np.ndarray) -> float:
        return float(np.sum(coef_A * thermal_output * thermal_output + coef_B * thermal_output + coef_C))

    def satisfy_load_constraints(thermal_output: np.ndarray, load_demand: float, tol: float = 0.05) -> bool:
        return float(np.sum(thermal_output)) >= (1.0 - tol) * float(load_demand)

    def satisfy_range_constraints(thermal_output: np.ndarray) -> bool:
        return bool(np.all(output_min <= thermal_output) and np.all(thermal_output <= output_max))

    def check_last_solutions(loss_list, last_nums: int) -> bool:
        if len(loss_list) < last_nums:
            return False
        last = loss_list[-last_nums:]
        return all(float(x) < 1.0 for x in last)

    def create_prompt(df_hist: pd.DataFrame, load_demand: float, num_sol: int) -> str:
        n = units
        Pd = float(load_demand)

        pmin_str = ", ".join([f"{v:.0f}" for v in output_min])
        pmax_str = ", ".join([f"{v:.0f}" for v in output_max])

        meta_start = (
            f"You need assistance in solving an optimization problem. "
            f"This problem involves {n} optimization variables, namely "
            f"{', '.join([f'p{i+1}' for i in range(n)])}. "
            f"These variables are subject to constraints defined by their minimum and maximum values "
            f"p_min=[{pmin_str}] and p_max=[{pmax_str}]. "
            f"Additionally, the sum of all p values must be greater than or equal to {Pd:.2f}. "
            f"Your objective is to provide values that minimize the objective value.\n\n"
            f"Below are some previous solution and their objective value pairs.\n"
        )

        solutions = ""
        if df_hist is not None and (not df_hist.empty):
            if num_sol > len(df_hist):
                num_sol = len(df_hist)
            for i in range(num_sol):
                idx = len(df_hist) - num_sol + i
                inputs = ", ".join([f"p{j+1}={df_hist[f'p{j+1}'].iloc[idx]:.3f}" for j in range(n)])
                val = float(df_hist["loss"].iloc[idx])
                solutions += f"input:\n{inputs}\nvalue:\n{val:.3f}\n\n"

        meta_end = (
            f"Give me a new ({', '.join([f'p{i+1}' for i in range(n)])}) pair that is different from all pairs above, "
            f"and has a value smaller than any of the above. The sum of all p values must be greater than or equal to {Pd:.2f}. "
            f"Do not give me any explanation. Output only in the form:\n"
            f"{', '.join([f'p{i+1}=<number>' for i in range(n)])}\n"
        )
        return meta_start + solutions + meta_end


    EQ_NUM_RE  = r"=\s*([-+]?(?:\d+(?:\.\d+)?)(?:[eE][-+]?\d+)?)"
    ANY_NUM_RE = r"[-+]?(?:\d+(?:\.\d+)?)(?:[eE][-+]?\d+)?"

    rows_gen, rows_cost = [], []
    time_results = []  
    
    scenarios = sorted(pf["scenario"].unique())

    for sid in scenarios:
        print(f"[GPTOPF] scenario {sid}", flush=True)
        t0 = time.time()  
        time_att1 = None
        time_att5 = None
        
        load_demand = float(load_demand_map.get(int(sid), 0.0))
        _ = get_active_branch_mask(int(sid))

        df_hist = pd.DataFrame(columns=["loss"] + [f"p{i+1}" for i in range(units)])
        loss_list = []
        cost_att1 = None

        for _ in range(3):
            text = create_prompt(df_hist, load_demand, 15)
            chat = client_.chat.completions.create(
                messages=[{"role": "user", "content": text}],
                model="gpt-4.1-nano",
            )
            txt = (chat.choices[0].message.content or "").strip()

            nums = re.findall(EQ_NUM_RE, txt)
            numbers = [float(x) for x in nums][:units]

            if len(numbers) != units:
                tail = txt.split("=")[-1]
                numbers = [float(x) for x in re.findall(ANY_NUM_RE, tail)][:units]

            if len(numbers) != units:
                numbers = [float(x) for x in re.findall(ANY_NUM_RE, txt)][:units]

            if len(numbers) != units:
                if check_last_solutions(loss_list, 3):
                    continue
                else:
                    continue

            thermal_ = np.clip(np.array(numbers, dtype=float), output_min, output_max)

            need = float(load_demand) - float(thermal_.sum())
            if need > 1e-9:
                headroom = output_max - thermal_
                room = float(headroom.sum())
                if room >= need + 1e-9:
                    w = np.divide(headroom, room, out=np.zeros_like(headroom), where=headroom > 0)
                    thermal_ = thermal_ + w * need

            if satisfy_range_constraints(thermal_) and satisfy_load_constraints(thermal_, load_demand, tol=0.05):
                loss = float(calculate(thermal_))
                loss_list.append(loss)
                if cost_att1 is None:
                    cost_att1 = loss
                    time_att1 = time.time() - t0   # ← 第一次成功时间
                new_row = {"loss": loss}
                for i in range(units):
                    new_row[f"p{i+1}"] = float(thermal_[i])

                df_hist = pd.concat([df_hist, pd.DataFrame([new_row])], ignore_index=True)
                df_hist.sort_values(by="loss", ascending=True, inplace=True)

        if df_hist.empty:
            continue

        best = df_hist.iloc[0]
        time_att3 = time.time() - t0 
        rows_gen.append({
            "scenario": int(sid),
            **{f"p{k+1}": float(best[f"p{k+1}"]) for k in range(units)}
        })
        rows_cost.append({
            "scenario": int(sid),
            "cost_att1": float(cost_att1) if cost_att1 is not None else np.nan,
            "cost_att3": float(best["loss"]),
        })
        time_results.append({
            "scenario": int(sid),
            "time_att1": time_att1,
            "time_att3": time_att3
        })
        
    pd.DataFrame(rows_cost).to_csv(
        os.path.join(SAVE_DIR, "opf_cost.csv"), index=False
    )
    pd.DataFrame(rows_gen).to_csv(
        os.path.join(SAVE_DIR, "opf_gen.csv"), index=False
    )
    pd.DataFrame(time_results).to_csv(
        os.path.join(SAVE_DIR, "opf_time.csv"), index=False
    )
    
    return {
        "model": "model_GPTOPF",
        "opf_gen": pd.DataFrame(rows_gen),
        "opf_cost": pd.DataFrame(rows_cost),
        "n_scenarios": len(rows_cost),
    }


In [11]:
MODEL_RUNNER = {
    "Model_ACOPF": run_acopf,
    "model_GridFM": lambda df: run_model_gridfm_predict_then_opf(),
    "model_GPTOPF": run_gptopf,
}

result = MODEL_RUNNER[model](df)
print("result:", result)


[GPTOPF] scenario 0
[GPTOPF] scenario 1
[GPTOPF] scenario 2
[GPTOPF] scenario 3
[GPTOPF] scenario 4
[GPTOPF] scenario 5
[GPTOPF] scenario 6
[GPTOPF] scenario 7
[GPTOPF] scenario 8
[GPTOPF] scenario 9
[GPTOPF] scenario 10
[GPTOPF] scenario 11
[GPTOPF] scenario 12
[GPTOPF] scenario 13
[GPTOPF] scenario 14
[GPTOPF] scenario 15
[GPTOPF] scenario 16
[GPTOPF] scenario 17
[GPTOPF] scenario 18
[GPTOPF] scenario 19
[GPTOPF] scenario 20
[GPTOPF] scenario 21
[GPTOPF] scenario 22
[GPTOPF] scenario 23
[GPTOPF] scenario 24
[GPTOPF] scenario 25
[GPTOPF] scenario 26
[GPTOPF] scenario 27
[GPTOPF] scenario 28
[GPTOPF] scenario 29
[GPTOPF] scenario 30
[GPTOPF] scenario 31
[GPTOPF] scenario 32
[GPTOPF] scenario 33
[GPTOPF] scenario 34
[GPTOPF] scenario 35
[GPTOPF] scenario 36
[GPTOPF] scenario 37
[GPTOPF] scenario 38
[GPTOPF] scenario 39
[GPTOPF] scenario 40
[GPTOPF] scenario 41
[GPTOPF] scenario 42
[GPTOPF] scenario 43
[GPTOPF] scenario 44
[GPTOPF] scenario 45
[GPTOPF] scenario 46
[GPTOPF] scenario 47
[G